# 解决偏移问题

有三种偏移  
1.协变量偏移：即输入的变量分布发生改变，但是判定逻辑没改变即P（y|x）   
2.标签偏移：标签变了，但表现形式没变即P(x|y)   
3.概念偏移：判定逻辑变了即P(y|x)

In [ ]:
# to handle covariate shift, we need a logistic regression
# no need to include labels

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_with_covariate_correction(x_source,y_source,x_target,model,main_optimizer):
    m ,d = x_source.shape
    n ,_ = x_target.shape

    x_combined = torch.cat([x_source, x_target],dim = 0)
    z_combined = torch.cat([torch.zeros(m),torch.ones(n)]).unsqueeze(1)

    prob_model = nn.Linear(d,1,device=device)
    prob_optim = optim.Adam(prob_model.parameters(),lr = 0.03)
    prob_loss = nn.BCEWithLogitsLoss()
    epochs = 100
    for _ in range(epochs):
        prob_optim.zero_grad()
        out = prob_model(x_combined)
        l = prob_loss(out, z_combined)
        # l.mean().backward()
        l.backward()
        prob_optim.step()

    with torch.no_grad():
        hx = prob_model(x_source)
        weights = torch.exp(hx)
        weights /= (weights.mean()+ 1e-8)
        # 去掉极端值
        weights = torch.clamp(weights,0.1, 10.0)

    model.train()
    main_optimizer.zero_grad()
    mout = model(x_source)

    criterion = nn.CrossEntropyLoss(reduction = 'none')
    row_loss = criterion(mout, y_source)
    weighted_loss = (row_loss*weights.squeeze()).mean()
    weighted_loss.backward()
    main_optimizer.step()

    return weighted_loss.item()
        

In [ ]:
# use confusion matrix to solve label shift
import torch
import torch.nn as nn

# confusion_matrics was the total output of the validation set trained on the source data.
# test pedictions was the output scores of the target data from the model trained on source data
def get_label_shift_wights(confusion_matrics,test_predictions,train_labels,num_classes):
    C = torch.tensor(confusion_matrics,dtype=torch.float32)
    mu = torch.tensor(test_predictions)

    # mu = C * p_target
    p_target = torch.linalg.solve(C, mu)
    # make sure that there is no non-positive scores
    p_target = torch.relu(p_target)
    # get possibilities
    p_target /= p_target.sum()

    counts = torch.bincount(train_labels, minlength=num_classes)
    p_scores = counts / counts.sum()

    beta = p_target / (p_scores + 1e-8)
 
    return beta

def modify_model(beta,model,inputs,labels,batch_weights):
    outputs = model(inputs)
    raw_loss = nn.CrossEntropyLoss(reduction='none')(outputs,labels)
    # match weights for every input according to the labels
    batch_weights = beta[labels]
    weighted_loss = (raw_loss*batch_weights).mean()
    optimizer = torch.optim.SGD(model.parameters(),lr = 0.0003)
    optimizer.zero_grad()
    weighted_loss.backward()
    optimizer.step()
    return weighted_loss.item()


In [ ]:
# quite easy to solve concept drift
# fine tuning will be enough
import torch
import torch.nn as nn
import torch.optim as optim

def handle_concept_dift(model,new_batch_data,new_batch_labels,lr=1e-4):
    optimizer = optim.SGD(model.parameters(),lr = lr)
    # 利用原有模型即可
    model.train()
    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(model(new_batch_data),new_batch_labels)
    loss.backward()
    optimizer.step()